# GraphRAG 笔记本：索引与检索

本笔记本演示在本地项目目录下调用 **GraphRAG Python API**（`graphrag.api`）完成 **建索引**（`build_index`）与 **全局/局部检索**（`global_search` / `local_search`）。

**官方文档**：[GraphRAG 文档](https://microsoft.github.io/graphrag/)

---

## 推荐阅读顺序

1. **环境与依赖**（下一节）：安装包、Jupyter 中启用 `asyncio`、配置 API 密钥（OpenAI 或兼容网关后台）。  
2. **项目路径与 `.env`**：解析 `PROJECT_PATH` 并 `load_dotenv`。  
3. **加载配置**：`load_config` 读取 `settings.yaml`。  
4. **（可选）API 自检**：用仓库里的 `api_client.py` 验证 OpenAI `chat/completions` 与密钥（与 GraphRAG 所用密钥应一致）。  
5. **Build Graph**：运行索引流水线（耗时较长，需有效 LLM/Embedding 配置）。  
6. **查看输出**：Parquet、LanceDB。  
7. **（可选）Neo4j 可视化**：需本机或 Docker Neo4j。  
8. **Global / Local Search**：在已有索引结果上提问。

---

## 与 `api_client.py` 的关系

与 [OpenAI API Reference](https://platform.openai.com/docs/api-reference) 对齐的**兼容网关**（如 LinkAPI）只需把主机名换成平台提供的 Base，路径仍为 `/v1/chat/completions`、`/v1/embeddings` 等。

| 项目 | 含义 |
|------|------|
| **`settings.yaml` 中的 `api_base`** | GraphRAG（LiteLLM）使用的 **API 根**：一般为 `https://<host>/v1`，**不要**写成 `.../chat/completions`。若平台文档给出的完整地址是 `https://<host>/v1/chat/completions`，则 `api_base` 取 `https://<host>/v1`。补全与嵌入在 `settings.yaml` 中可分别配置。 |
| **`api_client.py` 请求的 URL** | **完整** Chat Completions 地址：`https://<host>/v1/chat/completions`。可通过环境变量 **`GRAPHRAG_API_BASE`** 或 **`OPENAI_API_BASE`** 设为 `https://<host>/v1`，由客户端拼接为 `.../chat/completions`（与 GraphRAG 的 `api_base` 主机应对齐）。 |

- 鉴权：`Authorization: Bearer <api_key>`；密钥填 **`.env`** 中的 `GRAPHRAG_API_KEY`，与 GraphRAG 的 `${GRAPHRAG_API_KEY}` **保持一致**。若走兼容网关，密钥为**该平台管理后台**生成的 API Key；直连 OpenAI 则使用 [OpenAI 平台](https://platform.openai.com/) 的密钥。

```bash
GRAPHRAG_API_KEY=<平台或 OpenAI 后台创建的密钥>
# 可选，与 settings.yaml 中 api_base 主机一致：
# GRAPHRAG_API_BASE=https://api.linkapi.org/v1
```

笔记本会在解析出 `PROJECT_PATH` 后自动 `load_dotenv`，以便 `load_config` 能展开 `${GRAPHRAG_API_KEY}`。

## 环境与依赖

### 安装（在已激活的虚拟环境中执行）

```bash
pip install graphrag pandas pyarrow lancedb nest-asyncio python-dotenv requests neo4j
```

若尚未初始化 GraphRAG 项目，可在项目目录执行 `graphrag init --root .`（本仓库已包含 `settings.yaml`、`prompts/`、`input/` 时可跳过）。

### Jupyter 与 `asyncio`

GraphRAG API 多处使用 `await`。在经典 Jupyter Notebook 中需安装并启用 **nest-asyncio**（下一代码单元会 `apply()`）。

### 工作目录

将 Jupyter **当前工作目录**设为包含 `settings.yaml` 的文件夹（一般为 `graphrag_quickstart`），或保持在上级目录（笔记本会自动尝试 `graphrag_quickstart` 子目录）。

### 密钥与端点小结

| 用途 | 说明 |
|------|------|
| `api_client.py` | `POST` 到 `.../v1/chat/completions`；环境变量 `GRAPHRAG_API_BASE` / `OPENAI_API_BASE` 为 `.../v1` 根路径时由代码拼接；密钥为构造参数 `api_key`（通常来自 `GRAPHRAG_API_KEY`） |
| GraphRAG `settings.yaml` | `api_key: ${GRAPHRAG_API_KEY}`，并在各模型下配置 **`api_base`**（`/v1` 根路径，非 chat 完整路径）；未设置时默认 OpenAI 官方 |
| `.env` | `GRAPHRAG_API_KEY` 与自检、`APIClient` 一致；可选 `GRAPHRAG_API_BASE` 与网关主机对齐 |

### 代理与 GraphRAG（LiteLLM / httpx）

若系统或终端设置了 `HTTP_PROXY`、`HTTPS_PROXY`，**GraphRAG 调用模型与嵌入时也可能走同一代理**。若出现连接被重置、超时或 TLS 异常，可检查代理设置，或临时在无代理环境下重试。仅修改 `api_client.py` 不会自动改变 GraphRAG 的网络行为；索引与检索仍依赖上述环境。

In [39]:
import nest_asyncio

nest_asyncio.apply()

from pathlib import Path
from pprint import pprint
import pandas as pd

import graphrag.api as api
from graphrag.config.load_config import load_config
from graphrag.index.typing.pipeline_run_result import PipelineRunResult

In [40]:
from pathlib import Path
from dotenv import load_dotenv

# 项目根目录：须包含 settings.yaml、input/、prompts/（请将 Jupyter 工作目录设为 graphrag_quickstart）
_cwd = Path.cwd().resolve()
PROJECT_PATH = _cwd if (_cwd / "settings.yaml").exists() else (_cwd / "graphrag_quickstart").resolve()
if not (PROJECT_PATH / "settings.yaml").exists():
    raise FileNotFoundError(
        f"未找到 settings.yaml（已检查 {_cwd} 与 graphrag_quickstart 子目录）。请将工作目录设为包含该文件的目录。"
    )

# 从项目根目录加载 .env，供 settings.yaml 中的 ${GRAPHRAG_API_KEY} 等变量使用
load_dotenv(PROJECT_PATH / ".env")

PROJECT_DIRECTORY = str(PROJECT_PATH)
OUTPUT_DIR = PROJECT_PATH / "output"
LANCEDB_URI = OUTPUT_DIR / "lancedb"

In [41]:
# 加载 GraphRAG 配置（会读取 settings.yaml，并展开 .env 中的变量）
graphrag_config = load_config(Path(PROJECT_DIRECTORY))

### （可选）用 `api_client.py` 自检 API

与 [`api_client.py`](api_client.py) 相同方式调用 OpenAI 兼容 **chat/completions**（默认官方 `api.openai.com`，或 `.env` 中 `GRAPHRAG_API_BASE` 指向网关 `https://<host>/v1`）。若下面单元能收到模型回复，说明 **密钥与网络** 正常。GraphRAG 仍依赖 `settings.yaml` 中的 `api_key`、`api_base` 与模型名正确；若此处成功而索引仍报 401，请核对 YAML 与 `.env` 是否一致，以及密钥是否来自**当前所用平台**（兼容网关后台 key，或 OpenAI 官网 key）。

#### 故障排查

- **401 / 密钥无效**：核对 `.env` 与平台密钥，无多余空格；`settings.yaml` 的 `api_base` 与 `GRAPHRAG_API_BASE` 主机应一致。  
- **仅自检失败、load_config 正常**：多为 `api_client` 的 URL 与 `settings.yaml` 的 `api_base` 不一致（主机或是否多写 `/v1`）。  
- **嵌入失败、补全成功**：检查 `embedding_models` 的 `api_base` 与嵌入模型名是否在平台可用。  
- **额度或计费**：在 OpenAI 或兼容平台查看用量与账单。  
- **地区或服务可用性**：以平台文档与账号状态为准。  
- **代理问题**：若堆栈含 `_prepare_proxy` 或连接被重置，可检查 `HTTP(S)_PROXY`；必要时在下一单元使用 `APIClient(..., trust_env_proxy=False)` 尝试直连，或调整 `NO_PROXY`。

#### 若报错 `unexpected keyword argument 'trust_env_proxy'`

说明 Jupyter **内核仍缓存旧版** `api_client`。请 **重启内核** 后从顶部重新运行，或使用下一单元中的 `importlib.reload(api_client)` 强制重载。下一单元会打印 `api_client.__file__`，请确认路径指向本项目的 `graphrag_quickstart/api_client.py`。

自检使用的模型名 **`gpt-4o`** 与 `settings.yaml` 里 `default_completion_model.model` 保持一致。

In [42]:
import os
import requests
from dotenv import load_dotenv
from pathlib import Path

# 确保加载 .env（若已运行「项目路径」单元可省略 load_dotenv）
load_dotenv(Path.cwd() / "graphrag_quickstart" / ".env")

from api_client import APIClient

api_key = os.environ.get("GRAPHRAG_API_KEY", "").strip()
client = APIClient(api_key=api_key, trust_env_proxy=False)
SELF_CHECK_MODEL = "gpt-4o"

resp = requests.post(
    client.base_url,
    headers=client.headers,
    json={
        "model": SELF_CHECK_MODEL,
        "messages": [{"role": "user", "content": "hi"}],
    },
    timeout=30,
)
print("状态码:", resp.status_code)
print("响应体:", resp.text)
print("请求 URL:", client.base_url)

状态码: 200
响应体: {"choices":[{"content_filter_results":{"hate":{"filtered":false,"severity":"safe"},"protected_material_code":{"filtered":false,"detected":false},"protected_material_text":{"filtered":false,"detected":false},"self_harm":{"filtered":false,"severity":"safe"},"sexual":{"filtered":false,"severity":"safe"},"violence":{"filtered":false,"severity":"safe"}},"finish_reason":"stop","index":0,"logprobs":null,"message":{"annotations":[],"content":"Hello! How can I assist you today?","refusal":null,"role":"assistant"}}],"created":1774169815,"id":"chatcmpl-DM8u7p1ERP6KIB9hsncNA4wEdeODv","model":"gpt-4o-2024-08-06","object":"chat.completion","prompt_filter_results":[{"prompt_index":0,"content_filter_results":{"hate":{"filtered":false,"severity":"safe"},"jailbreak":{"filtered":false,"detected":false},"self_harm":{"filtered":false,"severity":"safe"},"sexual":{"filtered":false,"severity":"safe"},"violence":{"filtered":false,"severity":"safe"}}}],"system_fingerprint":"fp_e9b9b028d7","usage":

In [43]:
import importlib
import os
import sys

if str(PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_PATH))

import api_client

importlib.reload(api_client)
from api_client import APIClient

print("已加载 api_client:", api_client.__file__)

# 与 settings.yaml 中 default_completion_model.model 保持一致
SELF_CHECK_MODEL = "gpt-4o"

api_key = os.environ.get("GRAPHRAG_API_KEY", "").strip()
if not api_key:
    print("未检测到 GRAPHRAG_API_KEY：请确认 .env 位于 PROJECT_PATH 下，并已运行「项目路径」单元以执行 load_dotenv。")
else:
    # 可选：若环境变量中的 HTTP(S)_PROXY 导致连接问题，可改为 trust_env_proxy=False
    client = APIClient(api_key=api_key, trust_env_proxy=False)
    print("Chat Completions URL:", client.base_url)
    reply = client.chat_completion(
        [{"role": "user", "content": "请只回复：OK"}],
        model=SELF_CHECK_MODEL,
    )
    print("API 自检回复:", reply if reply else "(空响应，请检查返回体或模型名)")

已加载 api_client: /Users/yuelabpublic/Desktop/graphRAG/graphrag_quickstart/api_client.py
Chat Completions URL: https://api.linkapi.org/v1/chat/completions
API 自检回复: OK


## 构建索引（Build Graph）

对 `input/` 中文档跑完整流水线，生成 `output/` 下 Parquet 与 LanceDB 等产物。**耗时与费用**与文档量、模型调用次数相关。

若 `extract_graph` 失败且提示 *No entities detected*，常见原因包括：密钥无效、模型名错误、网络问题、输入语言与提示词/实体类型不匹配，或输入内容过短。可先完成上一节 **API 自检**，并确认 `settings.yaml` 与 `api_client.py` 使用同一密钥与兼容的模型名。

In [44]:
index_result: list[PipelineRunResult] = await api.build_index(config=graphrag_config)

# 每个元素对应流水线中的一个 workflow 及其成功/失败信息
for workflow_result in index_result:
    status = f"error\n{workflow_result.error}" if workflow_result.error else "success"
    print(f"Workflow Name: {workflow_result.workflow}\tStatus: {status}")

CancelledError: 

## 查看索引产物

成功建索引后，`output/` 下会有 `entities.parquet`、`communities.parquet`、`community_reports.parquet`、`text_units.parquet`、`relationships.parquet` 等。若某一步失败，部分文件可能缺失或为空，需先修复配置并重跑 **Build Graph**。

In [ ]:
entities = pd.read_parquet(OUTPUT_DIR / "entities.parquet")
entities

,id,human_readable_id,title,type,description,text_unit_ids,frequency,degree
0,fd8e13ac-3a98-4695-9f8a-f94bb636dcb4,0,STREETS DEPARTMENT,ORGANIZATION,The STREETS DEPARTMENT in Philadelphia is a vi...,[26aa49dea96eea41685fc1d12eb85430e57bfc6a3a37b...,6,6
1,928961f3-8926-4fd1-a22c-c067e713f49b,1,WEST MARKET ST,GEO,"An example of a street with wide sidewalks, de...",[26aa49dea96eea41685fc1d12eb85430e57bfc6a3a37b...,1,1
2,cdaf273e-603a-48c3-a70d-32bdf0231d77,2,ARCH ST AT CONVENTION CENTER,GEO,A location known for its sidewalks and proximi...,[26aa49dea96eea41685fc1d12eb85430e57bfc6a3a37b...,1,1
3,16ef8c01-f172-4d39-a280-732ae8149b34,3,PHILADELPHIA COMPLETE STREETS DESIGN HANDBOOK,EVENT,The Philadelphia Complete Streets Design Handb...,[26aa49dea96eea41685fc1d12eb85430e57bfc6a3a37b...,2,7
4,ae8b174e-2eb4-4efa-9aaa-9dd03c231334,4,PHILADELPHIA PEDESTRIAN & BICYCLE PLAN,DOCUMENT,A resource for planning pedestrian and bicycle...,[26aa49dea96eea41685fc1d12eb85430e57bfc6a3a37b...,1,6
5,af970b31-e3bd-4917-87c0-e38b4a9dad66,5,AASHTO GUIDE,DOCUMENT,"A guide for planning, designing, and operating...",[26aa49dea96eea41685fc1d12eb85430e57bfc6a3a37b...,1,0
6,418ddd72-3e36-47bc-8982-65ddb56f15db,6,PUBLIC RIGHT OF WAY ACCESS GUIDELINES,DOCUMENT,Guidelines referenced for ensuring accessible ...,[26aa49dea96eea41685fc1d12eb85430e57bfc6a3a37b...,1,2
7,56e74d96-550a-4f83-8658-edb18ee4f47c,7,DEPT. OF PARKS AND RECREATION,ORGANIZATION,An organization responsible for some aspects o...,[26aa49dea96eea41685fc1d12eb85430e57bfc6a3a37b...,1,0
8,e1aab70b-c116-46c9-82b8-bfe6dee8a63e,8,CONSIDERATIONS,EVENT,Design considerations include prioritizing wid...,[26aa49dea96eea41685fc1d12eb85430e57bfc6a3a37b...,1,1
9,2a8c1b69-8dc4-4ddf-94b5-370dd4f4c534,9,GREEN STREET OPPORTUNITIES,EVENT,Green street opportunities focus on integratin...,[26aa49dea96eea41685fc1d12eb85430e57bfc6a3a37b...,1,1


In [ ]:
communities = pd.read_parquet(OUTPUT_DIR / "communities.parquet")
community_reports = pd.read_parquet(OUTPUT_DIR / "community_reports.parquet")

community_reports

,id,human_readable_id,community,level,parent,children,title,summary,full_content,rank,rating_explanation,findings,full_content_json,period,size
0,5628086ba28a210a82f9ea6d6bab67e40c251e5a6987c7...,5,5,1,0,[],Philadelphia Streets Department and Urban Infr...,"The Philadelphia Streets Department, a central...",# Philadelphia Streets Department and Urban In...,6.5,The community's impact is moderate due to its ...,[{'explanation': 'The Streets Department serve...,"{\n ""title"": ""Philadelphia Streets Departme...",2026-03-22,4
1,310875a2eee668d5a362a09ae903dc1263dfcd98a0b738...,6,6,1,0,[],Philadelphia Urban Design Community,Philadelphia has proactively cultivated a pede...,# Philadelphia Urban Design Community\n\nPhila...,8.5,Philadelphia's efforts in enhancing urban desi...,[{'explanation': 'Philadelphia has demonstrate...,"{\n ""title"": ""Philadelphia Urban Design Com...",2026-03-22,8
2,87b6da66f067ba107402e2da46f0b81d2e2669ca9aaa08...,0,0,0,-1,"[5, 6]",Philadelphia Streets Department and Urban Infr...,The community centers on the Streets Departmen...,# Philadelphia Streets Department and Urban In...,8.5,The impact severity rating reflects the crucia...,[{'explanation': 'Philadelphia's Streets Depar...,"{\n ""title"": ""Philadelphia Streets Departme...",2026-03-22,12
3,be2c51ccdd5eb23a2a1694cf6d22f3edeb66b830003679...,1,1,0,-1,[],Philadelphia Street Design Community,The community revolves around the utilization ...,# Philadelphia Street Design Community\n\nThe ...,7.5,The impact severity rating is high due to the ...,[{'explanation': 'The Philadelphia Complete St...,"{\n ""title"": ""Philadelphia Street Design Co...",2026-03-22,8
4,1504d534a43a4c4110e0afdf45bb7abeb45b1d3c2c6061...,2,2,0,-1,[],PennDOT ADA Compliance and Community Infrastru...,"The community prioritizes accessibility, focus...",# PennDOT ADA Compliance and Community Infrast...,8.0,The impact severity rating is high due to the ...,"[{'explanation': 'PennDOT, or the Pennsylvania...","{\n ""title"": ""PennDOT ADA Compliance and Co...",2026-03-22,3
5,6bfee9cd53e90026ffedb4778946e8ff5e465bd44ecf39...,3,3,0,-1,[],Philadelphia Urban Planning Entities,The Philadelphia Urban Planning community cent...,# Philadelphia Urban Planning Entities\n\nThe ...,7.5,The impact severity is high due to the critica...,[{'explanation': 'The Philadelphia Code provid...,"{\n ""title"": ""Philadelphia Urban Planning E...",2026-03-22,4
6,b782ef2d66da92340872b7250275e95512b298154a3ee5...,4,4,0,-1,[],Philadelphia Planning and Zoning Framework,The Philadelphia Planning and Zoning Framework...,# Philadelphia Planning and Zoning Framework\n...,6.7,The impact severity rating reflects significan...,[{'explanation': 'The Philadelphia Pedestrian ...,"{\n ""title"": ""Philadelphia Planning and Zon...",2026-03-22,4


In [ ]:
import lancedb

db = lancedb.connect(str(LANCEDB_URI))
# List available tables:
db.table_names()


['community_full_content', 'entity_description', 'text_unit_text']

In [ ]:
entity_table = db.open_table('entity_description')
entity_df = entity_table.to_pandas()

entity_df.head()

,id,vector,create_date,update_date,create_date_year,create_date_month,create_date_month_name,create_date_day,create_date_day_of_week,create_date_hour,create_date_quarter,update_date_year,update_date_month,update_date_month_name,update_date_day,update_date_day_of_week,update_date_hour,update_date_quarter
0,fd8e13ac-3a98-4695-9f8a-f94bb636dcb4,"[-0.018404143, -0.0028471397, -0.015464692, -0...",2026-03-22T07:50:35.958228+00:00,None,2026,3,March,22,Sunday,7,1,NaN,NaN,None,NaN,None,NaN,NaN
1,928961f3-8926-4fd1-a22c-c067e713f49b,"[0.010225976, 0.02181432, -0.02281558, -0.0022...",2026-03-22T07:50:35.958357+00:00,None,2026,3,March,22,Sunday,7,1,NaN,NaN,None,NaN,None,NaN,NaN
2,cdaf273e-603a-48c3-a70d-32bdf0231d77,"[0.021890463, 0.043748032, -0.019818187, -0.01...",2026-03-22T07:50:35.958440+00:00,None,2026,3,March,22,Sunday,7,1,NaN,NaN,None,NaN,None,NaN,NaN
3,16ef8c01-f172-4d39-a280-732ae8149b34,"[-0.007240121, -0.0026714301, -0.0149890855, -...",2026-03-22T07:50:35.958524+00:00,None,2026,3,March,22,Sunday,7,1,NaN,NaN,None,NaN,None,NaN,NaN
4,ae8b174e-2eb4-4efa-9aaa-9dd03c231334,"[-0.020584654, -0.010453035, -0.022778673, 0.0...",2026-03-22T07:50:35.958818+00:00,None,2026,3,March,22,Sunday,7,1,NaN,NaN,None,NaN,None,NaN,NaN


## 导入 Neo4j 可视化（可选）

**可选步骤**：仅用于在 Neo4j Browser 中查看图结构。跳过本节不影响 GraphRAG 索引与检索。

凭证通过环境变量：`NEO4J_USER`、`NEO4J_PASSWORD`；可选 `NEO4J_URI`（默认 `neo4j://localhost:7687`）。可与项目根目录 `.env` 或 shell `export` 配合。Parquet 路径使用上文已定义的 `OUTPUT_DIR`。

请先启动 Neo4j，再按顺序运行本节代码单元。

In [1]:
import os

from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
load_dotenv(env_path)
# 调试
print("cwd:", Path.cwd())
print("env exists:", env_path.exists())
print("NEO4J_USER:", "已设置" if os.environ.get("NEO4J_USER") else "未设置")
print("NEO4J_PASSWORD:", "已设置" if os.environ.get("NEO4J_PASSWORD") else "未设置")

cwd: /Users/yuelabpublic/Desktop/graphRAG/graphrag_quickstart
env exists: True
NEO4J_USER: 已设置
NEO4J_PASSWORD: 已设置


In [2]:
import os

from dotenv import load_dotenv
from pathlib import Path

# 在此单元中重新加载 .env，确保 NEO4J_* 变量已生效
_env = Path.cwd() / ".env"
if not _env.exists():
    _env = Path.cwd() / "graphrag_quickstart" / ".env"
load_dotenv(_env)

from neo4j import GraphDatabase

uri = os.environ.get("NEO4J_URI", "neo4j://localhost:7687")
username = os.environ.get("NEO4J_USER", "")
password = os.environ.get("NEO4J_PASSWORD", "")

if not username or not password:
    raise RuntimeError(
        "未设置 NEO4J_USER / NEO4J_PASSWORD。不需要图可视化时请跳过本节代码单元；"
        "需要连接时请配置环境变量后再运行。"
    )

driver = GraphDatabase.driver(uri, auth=(username, password))

In [3]:
import numpy as np
import time

NEO4J_DATABASE = "neo4j"


In [4]:
import pandas as pd
from pathlib import Path
# 项目路径（若已运行「项目路径」单元可删除以下两行）
_cwd = Path.cwd().resolve()
PROJECT_PATH = _cwd if (_cwd / "settings.yaml").exists() else (_cwd / "graphrag_quickstart").resolve()
OUTPUT_DIR = PROJECT_PATH / "output"

entities_df = pd.read_parquet(OUTPUT_DIR / "entities.parquet")
relationships_df = pd.read_parquet(OUTPUT_DIR / "relationships.parquet")
text_units_df = pd.read_parquet(OUTPUT_DIR / "text_units.parquet")
communities_df = pd.read_parquet(OUTPUT_DIR / "communities.parquet")
community_report_df = pd.read_parquet(OUTPUT_DIR / "community_reports.parquet")



In [5]:
import numpy as np

def run_cypher_batch(tx, statement, params_list):
    """Run a batch of Cypher statements safely."""
    for params in params_list:
        tx.run(statement, **params)

# ---------------- Create Entity Nodes ----------------
# GraphRAG entities.parquet 不含 x, y 列，故不写入坐标
entity_stmt = """
MERGE (e:Entity {id: $id})
SET e.human_readable_id = $human_readable_id,
    e.title = $title,
    e.description = $description,
    e.type = $type,
    e.frequency = $frequency,
    e.degree = $degree
"""

entity_params = entities_df.to_dict("records")
with driver.session() as session:
    session.execute_write(run_cypher_batch, entity_stmt, entity_params)

# ---------------- Create TextUnit Nodes ----------------
# 仅使用 text_units.parquet 实际存在的列（id, document_id, text, n_tokens）
# 若使用 extract_graph_nlp 或 base text_units，可能无 human_readable_id/entity_ids/relationship_ids/covariate_ids
textunit_stmt = """
MERGE (t:TextUnit {id: $id})
SET t.text = $text,
    t.n_tokens = $n_tokens,
    t.document_id = $document_id
"""

textunit_params = text_units_df[["id", "text", "n_tokens", "document_id"]].to_dict("records")
with driver.session() as session:
    session.execute_write(run_cypher_batch, textunit_stmt, textunit_params)

# ---------------- Link Entities to TextUnits ----------------
entity_to_textunit_stmt = """
MATCH (e:Entity {id: $entity_id})
MATCH (t:TextUnit {id: $text_unit_id})
MERGE (e)-[:CREATED_FROM]->(t)
"""

entity_to_textunit_params = []

for _, row in entities_df.iterrows():
    tu_ids = row.get("text_unit_ids")
    # Make sure tu_ids is a list
    if isinstance(tu_ids, np.ndarray) and len(tu_ids) > 0:
        for tu_id in tu_ids:
            entity_to_textunit_params.append({
                "entity_id": row["id"],
                "text_unit_id": tu_id
            })

with driver.session() as session:
    session.execute_write(run_cypher_batch, entity_to_textunit_stmt, entity_to_textunit_params)

# ---------------- Create Communities ----------------
community_stmt = """
MERGE (c:Community {id: $id})
SET c.human_readable_id = $human_readable_id,
    c.community = $community,
    c.level = $level,
    c.parent = $parent,
    c.children = $children,
    c.title = $title,
    c.period = $period,
    c.size = $size
"""

community_params = communities_df.to_dict("records")
with driver.session() as session:
    session.execute_write(run_cypher_batch, community_stmt, community_params)

# ---------------- Link Entities to Communities ----------------
entity_to_community_stmt = """
MATCH (e:Entity {id: $entity_id})
MATCH (c:Community {id: $community_id})
MERGE (e)-[:IN_COMMUNITY]->(c)
"""

entity_to_community_params = []

for _, row in communities_df.iterrows():
    ent_ids = row.get("entity_ids")
    if isinstance(ent_ids, np.ndarray) and len(ent_ids) > 0:
        for eid in ent_ids:
            entity_to_community_params.append({
                "entity_id": eid,
                "community_id": row["id"]
            })

with driver.session() as session:
    session.execute_write(run_cypher_batch, entity_to_community_stmt, entity_to_community_params)


# ---------------- Link TextUnits to Communities ----------------
textunit_to_community_stmt = """
MATCH (t:TextUnit {id: $text_unit_id})
MATCH (c:Community {id: $community_id})
MERGE (t)-[:PART_OF]->(c)
"""

textunit_to_community_params = []

for _, row in communities_df.iterrows():
    tu_ids = row.get("text_unit_ids")
    if isinstance(tu_ids, np.ndarray) and len(tu_ids) > 0:
        for tu_id in tu_ids:
            textunit_to_community_params.append({
                "text_unit_id": tu_id,
                "community_id": row["id"]
            })

with driver.session() as session:
    session.execute_write(run_cypher_batch, textunit_to_community_stmt, textunit_to_community_params)


# ---------------- Create Relationships Between Entities ----------------
entity_rel_stmt = """
MATCH (start:Entity {title: $start})
MATCH (end:Entity {title: $end})
MERGE (start)-[:RELATED {weight: $weight, description: $description}]->(end)
"""

rel_params_list = []

for _, row in relationships_df.iterrows():
    if pd.notna(row["source"]) and pd.notna(row["target"]):
        rel_params_list.append({
            "start": row["source"],
            "end": row["target"],
            "weight": row.get("weight"),
            "description": row.get("description")
        })

with driver.session() as session:
    session.execute_write(run_cypher_batch, entity_rel_stmt, rel_params_list)


# ---------------- Add Community Reports ----------------
community_report_stmt = """
MATCH (c:Community {community: $community})
SET
  c.summary = $summary,
  c.full_content = $full_content,
  c.rank = $rank,
  c.rating_explanation = $rating_explanation
WITH c
UNWIND range(0, size($findings) - 1) AS finding_idx
WITH c, finding_idx, $findings[finding_idx] AS finding
MERGE (c)-[:HAS_FINDING]->(f:Finding {
  community: $community,
  index: finding_idx
})
SET f += finding
"""


report_params = community_report_df.rename(columns={"id": "community_id"}).to_dict("records")
with driver.session() as session:
    session.execute_write(run_cypher_batch, community_report_stmt, report_params)

print("✅ GraphRAG import into Neo4j complete!")


✅ GraphRAG import into Neo4j complete!


导入完成后，可在 **Neo4j Browser** 中执行 Cypher 浏览 `Entity`、`Community`、`TextUnit` 等节点与关系。

## 全局检索（Global Search）

基于社区摘要等做 **宏观归纳**，适合跨文档、概括性问题。查询阶段仍会调用 `settings.yaml` 中配置的补全模型（`api_key` / `api_base` / `gpt-4o` 等与索引阶段一致）。

下方示例问题为英文演示，可按你的语料改写 `question`。

In [ ]:
question = (
    "Summarize the main sidewalk width requirements, ADA/clear-path considerations, "
    "and green-street opportunities described across the TREATMENT 4.3 sections "
    "of the Complete Streets handbook excerpts."
)

In [ ]:
response, context = await api.global_search(
    config=graphrag_config,
    entities=entities,
    communities=communities,
    community_reports=community_reports,
    community_level=2,
    dynamic_community_selection=False,
    response_type="Multiple Paragraphs",
    query=question,
)

In [ ]:
print(response)

# Summary of Requirements and Opportunities in TREATMENT 4.3 Sections

## Sidewalk Width Requirements
The Complete Streets Design Handbook prioritizes accommodating high pedestrian traffic by emphasizing adequate sidewalk width. This aspect is crucial for areas with dense foot traffic and significant commercial activity, ensuring that pedestrians can navigate comfortably and safely through urban spaces. Wider sidewalks are essential to mitigate congestion and enhance overall pedestrian accessibility [Data: Reports (1, 6, +more)].

## ADA Compliance and Clear-Path Considerations
ADA compliance remains a critical aspect of the Complete Streets Design Handbook, especially within the TREATMENT 4.3 sections. Strict guidelines ensure that sidewalks and any modifications thereof provide clear paths for individuals with disabilities. Critical elements such as curb ramps need to adhere to these standards, thereby fostering an inclusive urban environment. Emphasizing these considerations ensures

## 局部检索（Local Search）

结合实体、关系与文本块做 **细粒度问答**，适合指向具体事实的问题。同样需要有效的嵌入与补全模型配置（嵌入走 `embedding_models`，补全走 `completion_models`）。

可按需修改 `local_question`，并确保前文已成功加载 `entities`、`communities`、`community_reports` 以及本节的 `text_units`、`relationships`。

In [ ]:
local_question = (
    "What minimum sidewalk widths are required on local streets versus walkable commercial corridors, "
    "and what maximum cross slope is specified for sidewalks?"
)

In [ ]:
text_units = pd.read_parquet(OUTPUT_DIR / "text_units.parquet")
relationships = pd.read_parquet(OUTPUT_DIR / "relationships.parquet")

In [ ]:
local_response, local_context = await api.local_search(
    config=graphrag_config,
    entities=entities,
    communities=communities,
    community_reports=community_reports,
    text_units=text_units,
    relationships=relationships,
    covariates=None,
    community_level=3,
    response_type="Multiple Paragraphs",
    query=local_question,
)

In [ ]:
print(local_response)

### Minimum Sidewalk Width Requirements

The Philadelphia Complete Streets Design Handbook specifies distinct minimum sidewalk widths required for different types of streets within Philadelphia. On **local streets**, the handbook mandates a minimum total sidewalk width of **10 feet** [Data: Sources (0)]. In contrast, when it comes to **walkable commercial corridors**, the required minimum width increases to **12 feet**. These specifications take into account the different levels of pedestrian traffic and activity that characterize each street type [Data: Sources (0); Entities (3)].

### Maximum Cross Slope

The Complete Streets Design Handbook also addresses the specifications for the cross slope of sidewalks to ensure accessibility and comfort for pedestrians. The **maximum cross slope** permitted for sidewalks is **2.0%** over a width of at least **5 feet**. This requirement is in place to comply with the Americans with Disabilities Act (ADA) standards, facilitating easy navigation f